# RAG Evaluation with RAGAS
## A Hands-On Workshop

Building a RAG pipeline is only half the job. How do you know it is actually working well?
**RAGAS** (Retrieval Augmented Generation Assessment) gives you a suite of automatic metrics that score your pipeline without needing manually labelled data for every question.

By the end of this workbook you will:
- Build a simple RAG pipeline to evaluate
- Understand three core RAGAS metrics: **Faithfulness**, **Answer Relevance**, and **Context Recall**
- Run an evaluation and read the scores
- Know how to diagnose and improve a low-scoring pipeline

In [ ]:
# Install dependencies -- only runs on Google Colab.
# Local users: pip install ragas langchain-openai langchain-chroma python-dotenv
import sys
if 'google.colab' in sys.modules:
    %pip install -q ragas langchain-openai langchain-chroma langchain-community python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))

---

## Why Evaluate a RAG Pipeline?

RAG pipelines can fail in ways that are invisible from the outside:

| Failure mode | Symptom | Metric that catches it |
|---|---|---|
| LLM makes up facts not in the context | Confident but wrong answers | **Faithfulness** |
| Answer doesn't address the question | Technically true but irrelevant | **Answer Relevance** |
| Retriever misses necessary documents | Incomplete answers | **Context Recall** |
| Retriever pulls noisy unrelated chunks | Distracted LLM reasoning | **Context Precision** |

RAGAS scores each metric from 0 to 1. A score of 1 is perfect.
Running evaluation regularly — not just once — catches regressions when you change your chunking, model, or prompts.

---

## Part 1 -- Build the RAG Pipeline Under Test

We need a working RAG pipeline before we can evaluate it.
We will use a small hardcoded knowledge base about Python programming so the answers are easy to verify by hand.

In [ ]:
from langchain_core.documents import Document

# Knowledge base: 6 short documents about Python
DOCS = [
    Document(page_content="Python lists are ordered, mutable sequences. They support indexing, slicing, and methods like append(), extend(), and pop().", metadata={"source": "python-basics"}),
    Document(page_content="Python dictionaries store key-value pairs. Keys must be hashable. Common methods include get(), keys(), values(), items(), and update().", metadata={"source": "python-basics"}),
    Document(page_content="Python functions are defined with the def keyword. They support default arguments, *args for variable positional arguments, and **kwargs for keyword arguments.", metadata={"source": "python-functions"}),
    Document(page_content="List comprehensions provide a concise way to create lists: [expr for item in iterable if condition]. They are generally faster than equivalent for loops.", metadata={"source": "python-advanced"}),
    Document(page_content="Python classes are defined with the class keyword. The __init__ method initializes instances. Python supports single and multiple inheritance.", metadata={"source": "python-oop"}),
    Document(page_content="Python exceptions are handled with try/except/finally blocks. Common exceptions include ValueError, TypeError, KeyError, and IndexError.", metadata={"source": "python-errors"}),
]

print(f"Knowledge base: {len(DOCS)} documents")
for doc in DOCS:
    print(f"  [{doc.metadata['source']}] {doc.page_content[:60]}...")

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Build an in-memory vector store from the documents
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(DOCS, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Quick test -- confirm retrieval works before evaluation
test_results = retriever.invoke("How do Python lists work?")
print(f"Retrieved {len(test_results)} chunks for test query")
for r in test_results:
    print(f"  {r.page_content[:80]}...")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context below. If the context does not contain the answer, say "I don't know".

Context:
{context}

Question: {question}
Answer:""".strip())

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Smoke test
answer = rag_chain.invoke("What methods do Python lists support?")
print("Answer:", answer)